# Implementing and Checking a Binary Heap


```{contents}
:local:
:depth: 2
```


```{index} Repair Only the Part an Operation Can Disturb
```



```{index} Repair Only the Part an Operation Can Disturb
```

## Repair Only the Part an Operation Can Disturb



Enqueue adds a new entry at the next open leaf position so the complete shape is preserved. All old parent-child relations are unchanged. Only the new entry's path toward the root can violate min-heap order. Bubble-up compares with the parent and swaps while the child is smaller. Moving upward strictly decreases its index, so repair terminates.

Start with [2, 5, 4] and enqueue 1. Appending produces [2, 5, 4, 1]. Comparing 1 with parent 5 gives [2, 1, 4, 5]. Comparing that 1 with root 2 gives [1, 2, 4, 5]. Do not sort the whole array; that would enforce a stronger and more expensive condition than the operation needs.

Dequeue first saves the root result. Move the last live entry to the root and reduce Count. The only possible ordering defects now lie downward from that replacement. Choose the smaller existing child before swapping; exchanging with an arbitrary child can leave a smaller sibling above a larger parent relation.

| Heap after removing 1 | Inspection | Next step |
| --- | --- | --- |
| [5, 2, 4] | Children 2 and 4 | Choose 2 |
| [2, 5, 4] | 5 has no child | Stop |

A singleton removal needs no child access after Count becomes zero. An empty removal must follow the stated failure policy without first reading position zero. Also clear unused reference slots when the implementation owns a backing array and should release references to removed objects. Reducing Count hides a slot from the abstract sequence, but it does not itself erase a stored reference.

The height of a complete heap is logarithmic in its live size, which bounds the number of repair levels. Array growth can add copying to an enqueue, so qualify an operation's cost rather than ignoring representation maintenance. Bulk building by bottom-up repair is a related algorithm with a different aggregate argument; it is not proved merely by multiplying one operation's bound.

Guided exercise: repeat enqueue with a value 6 that already belongs below its parent. Predict how many swaps occur and distinguish swaps from comparisons. Then remove down to empty and enqueue again. Those cases test both the invariant and whether endpoint handling leaves the structure reusable.

For related explanation, see the further reading below.[^development-reading]


```{index} Enqueue and Bubble Up
```

## Enqueue and Bubble Up




```{index} heap; bubble up
```


To insert into a heap, place the new item at the end to preserve the complete-tree shape, then **bubble up** while it violates the heap order rule.


In [1]:
using System;
using System.Collections.Generic;

var heap = new List<int> { 1, 3, 2, 7, 5, 6, 4 };
heap.Add(0);

int child = heap.Count - 1;
while (child > 0)
{
    int parent = (child - 1) / 2;
    if (heap[parent] <= heap[child])
    {
        break;
    }

    (heap[parent], heap[child]) = (heap[child], heap[parent]);
    child = parent;
}

Console.WriteLine(string.Join(", ", heap));


0, 1, 2, 3, 5, 6, 4, 7


The new value `0` starts at the end. It swaps with larger parents until the min-heap property is restored.


**Common mistake.** Computing the parent as `child / 2` instead of `(child - 1) / 2`. For index 1 both give 0, which hides the bug, but for index 2 the wrong formula gives 1 instead of 0, and the new value is compared with a sibling rather than its parent.


Trace the appended zero from index seven to the root:

| Stage | Index path | Array |
| --- | --- | --- |
| Append | 7 | 1, 3, 2, 7, 5, 6, 4, 0 |
| Swap with 7 | 7 → 3 | 1, 3, 2, 0, 5, 6, 4, 7 |
| Swap with 3 | 3 → 1 | 1, 0, 2, 3, 5, 6, 4, 7 |
| Swap with 1 | 1 → 0 | 0, 1, 2, 3, 5, 6, 4, 7 |

Before insertion, every old parent-child relation satisfies the order rule. Appending adds only one possible violation between the new item and its parent. Each swap repairs that relation and moves the possible violation upward. Stop at the root or when the parent is no larger. This explains why repairing one path is sufficient.

```{index} Code Tracing: Enqueue Four Priorities
```

### Code Tracing: Enqueue Four Priorities

Starting from an empty min-heap, enqueue 9, 4, 7, 1 in that order. After each enqueue write the array, then write it again after bubble-up finishes, with the number of swaps that enqueue needed. Predict the final array and the total number of swaps before running the solution.

In [ ]:
// Solution
using System;
using System.Collections.Generic;

var heap = new List<int>();
int totalSwaps = 0;
foreach (int value in new[] { 9, 4, 7, 1 })
{
    heap.Add(value);
    string appended = string.Join(", ", heap);
    int child = heap.Count - 1, swaps = 0;
    while (child > 0)
    {
        int parent = (child - 1) / 2;
        if (heap[parent] <= heap[child]) break;
        (heap[parent], heap[child]) = (heap[child], heap[parent]);
        child = parent;
        swaps++;
    }
    totalSwaps += swaps;
    Console.WriteLine($"enqueue {value}: appended [{appended}] -> after {swaps} swap(s) [{string.Join(", ", heap)}]");
}
Console.WriteLine($"total swaps = {totalSwaps}");

enqueue 9: appended [9] -> after 0 swap(s) [9]
enqueue 4: appended [9, 4] -> after 1 swap(s) [4, 9]
enqueue 7: appended [4, 9, 7] -> after 0 swap(s) [4, 9, 7]
enqueue 1: appended [4, 9, 7, 1] -> after 2 swap(s) [1, 4, 7, 9]
total swaps = 3


The final array is `1, 4, 7, 9` after three swaps in total. The 4 swaps once, past 9. The 7 lands at index 2 under parent 4 and needs no swap. The 1 is appended at index 3, whose parent is index 1 holding 9, so it swaps once to index 1 and once more past 4 to the root. The final array happens to be sorted, which is a coincidence of this input, not a property of heaps; enqueue 3 next and the array becomes `1, 3, 7, 9, 4`, valid and unsorted.

```{index} Dequeue and Sink Down
```

## Dequeue and Sink Down




```{index} heap; sink down
```


To remove the minimum item, remove the root, move the last item to the root, then **sink down** until the heap order rule is restored.


In [1]:
using System;
using System.Collections.Generic;

var heap = new List<int> { 1, 3, 2, 7, 5, 6, 4 };
int removed = heap[0];
heap[0] = heap[^1];
heap.RemoveAt(heap.Count - 1);

int parent = 0;
while (true)
{
    int left = 2 * parent + 1;
    int right = 2 * parent + 2;
    int smallest = parent;

    if (left < heap.Count && heap[left] < heap[smallest]) smallest = left;
    if (right < heap.Count && heap[right] < heap[smallest]) smallest = right;
    if (smallest == parent) break;

    (heap[parent], heap[smallest]) = (heap[smallest], heap[parent]);
    parent = smallest;
}

Console.WriteLine($"Removed: {removed}");
Console.WriteLine(string.Join(", ", heap));


Removed: 1
2, 3, 4, 7, 5, 6


Both bubble-up and sink-down follow one root-to-leaf path, so their work grows with heap height, not with every item in the heap.


**Common mistake.** Swapping with the left child whenever it is smaller than the parent, without checking the right child. If the right child is smaller still, the left child ends up above a smaller sibling and the order rule is broken one level down.


In the removal example, replacing root 1 with the last value 4 gives `4, 3, 2, 7, 5, 6`. Compare both children before choosing the swap: 2 is smaller than 3, so swap indexes zero and two. The result is `2, 3, 4, 7, 5, 6`. The next child is 6; 4 is already no larger, so stop.

Choosing a larger child can leave the root larger than the other child. Sink-down must choose the smaller existing child and handle a last parent with only a left child. Removing the last array position preserves the complete shape. For a singleton, removal leaves no root to repair.

The complete generic heap below hides the backing list and accepts an optional comparer. The element type must support the default ordering, or you must supply a comparer. `Peek` and `Dequeue` throw on empty; `TryDequeue` distinguishes empty from a stored default value. Equal values are legal, but this heap does not impose stable arrival order on equal comparisons.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var priorities = new BinaryMinHeap<int>();
foreach (int urgency in new[] { 3, 0, 2, 0, 5 }) priorities.Enqueue(urgency);
Console.WriteLine($"count={priorities.Count}, minimum={priorities.Peek()}, valid={priorities.IsValid()}");
Console.WriteLine("raw=" + string.Join(", ", priorities.Snapshot()));
while (priorities.TryDequeue(out int urgency)) Console.WriteLine(urgency);
priorities.Enqueue(0);
Console.WriteLine($"reused zero={priorities.Dequeue()}, empty={priorities.Count == 0}");

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}

count=5, minimum=0, valid=True
raw=0, 0, 2, 3, 5
0
0
2
3
5
reused zero=0, empty=True


`List<T>` grows its storage when necessary. Bubble-up uses O(log n) comparisons, but a single enqueue that copies a full backing array can take O(n) time. Across a long sequence, storage growth contributes amortized O(1) per append; enqueue therefore has amortized O(log(n + 1)) time. Dequeue repairs at most one downward path and takes O(log(n + 1)); it does not shrink this backing list. Peek and Count take O(1). The iterative repair uses O(1) extra space, while the stored items use O(n) space.

`IsValid` scans all parent-child relations, so it takes Θ(n) time. It is a diagnostic method, not part of every production operation. `Snapshot` also takes Θ(n) time and space to copy the occupied values. Checking the minimum alone would miss a violation inside a subtree. Test empty and singleton states, duplicate priorities, an odd final child, ascending/descending input, draining, and reuse.

A heap can also be built from an existing array by sinking internal nodes from right to left. Most nodes are near leaves and require little repair; summing that work gives O(n) bottom-up construction. Repeated insertion has an O(n log n) worst-case bound. This chapter implements repeated enqueue first; bottom-up construction is an extension.

```{index} Build a Heap Bottom Up
```

### Build a Heap Bottom Up

When all n values are available at once, there is a faster way to make them a heap than enqueueing them one at a time. Place the values in the array in any order, then walk the internal nodes from the last one, index n / 2 - 1, back to index 0, sinking each one down as `Dequeue` does. When the loop reaches a node, both of its subtrees have already been repaired, so one sink-down makes the subtree rooted there a heap; when the loop finishes at index 0, the whole array is a heap.

The cost argument is the interesting part. Repeated enqueue costs O(log n) per item because the item may travel all the way up from a leaf, and about half of all items *are* leaves. Bottom-up construction never touches the leaves, and the nodes it does touch are mostly near the bottom with short distances to sink: about n / 4 nodes can sink at most one level, n / 8 at most two, n / 16 at most three, and so on. The sum n / 4 × 1 + n / 8 × 2 + n / 16 × 3 + ... is less than n, so building a heap bottom up is O(n), compared with O(n log n) for n enqueues. The example counts swaps for both methods on the same input.

In [ ]:
using System;
using System.Linq;

foreach (int n in new[] { 15, 1023, 65535 })
{
    int[] descending = Enumerable.Range(1, n).Reverse().ToArray();   // worst case for a min-heap
    Console.WriteLine($"n={n,6}: repeated enqueue swaps={EnqueueAll(descending),8}   bottom-up swaps={BuildBottomUp(descending),6}   n log2 n={(int)(n * Math.Log2(n)),8}");
}

static int EnqueueAll(int[] values)
{
    int[] heap = new int[values.Length];
    int swaps = 0;
    for (int count = 0; count < values.Length; count++)
    {
        heap[count] = values[count];
        for (int child = count; child > 0; child = (child - 1) / 2)
        {
            int parent = (child - 1) / 2;
            if (heap[parent] <= heap[child]) break;
            (heap[parent], heap[child]) = (heap[child], heap[parent]);
            swaps++;
        }
    }
    return swaps;
}

static int BuildBottomUp(int[] values)
{
    int[] heap = (int[])values.Clone();
    int n = heap.Length, swaps = 0;
    for (int i = n / 2 - 1; i >= 0; i--)        // last internal node back to the root
    {
        int parent = i;
        while (true)
        {
            int left = 2 * parent + 1, right = left + 1, smallest = parent;
            if (left < n && heap[left] < heap[smallest]) smallest = left;
            if (right < n && heap[right] < heap[smallest]) smallest = right;
            if (smallest == parent) break;
            (heap[parent], heap[smallest]) = (heap[smallest], heap[parent]);
            parent = smallest;
            swaps++;
        }
    }
    return swaps;
}

n=    15: repeated enqueue swaps=      34   bottom-up swaps=    11   n log2 n=      58
n=  1023: repeated enqueue swaps=    8194   bottom-up swaps=  1013   n log2 n=   10228
n= 65535: repeated enqueue swaps=  917506   bottom-up swaps= 65519   n log2 n= 1048558


On descending input, the worst case for repeated enqueue, the swap counts for enqueueing grow like n log2 n, while the bottom-up counts stay below n at every size: for 65,535 values about 65,000 swaps against almost one million. The two methods produce different valid heaps from the same input, which is fine, because the contract is the heap property, not a specific layout. Use bottom-up construction when the data arrives as a batch, for example when a scheduler loads the day's jobs; use enqueue when items trickle in. The library's `PriorityQueue<TElement, TPriority>` constructor that takes a collection does the bottom-up build for you.

```{index} Change a Priority or Remove from the Middle
```

### Change a Priority or Remove from the Middle

Schedulers often need a third operation: an item already in the queue becomes more or less urgent, or is cancelled. A heap cannot find an item by its value, because the partial order gives no direction to search, so the application must remember the item's current index, typically in a dictionary from item to position that the swap code keeps up to date. Once the index is known, the repair is the one already written. If the priority decreased, the item may now be smaller than its parent, so bubble it up. If the priority increased, it may now be larger than a child, so sink it down. Exactly one of the two can be needed, because the item's old position satisfied the rule with both its parent and its children. Removal from the middle is the same idea: overwrite the slot with the last item, shrink the count, and repair that slot in whichever direction the replacement needs.

The example changes a priority by index in a small heap and shows both directions of repair.

In [ ]:
using System;
using System.Collections.Generic;

var heap = new List<int> { 1, 3, 2, 7, 5, 6, 4 };
Console.WriteLine($"start          [{string.Join(", ", heap)}]");

ChangePriority(heap, index: 4, newValue: 0);   // 5 becomes 0: must bubble up
Console.WriteLine($"index 4 -> 0   [{string.Join(", ", heap)}]  valid={IsValid(heap)}");

ChangePriority(heap, index: 0, newValue: 9);   // root becomes 9: must sink down
Console.WriteLine($"index 0 -> 9   [{string.Join(", ", heap)}]  valid={IsValid(heap)}");

static void ChangePriority(List<int> heap, int index, int newValue)
{
    int old = heap[index];
    heap[index] = newValue;
    if (newValue < old) BubbleUp(heap, index);
    else SinkDown(heap, index);
}

static void BubbleUp(List<int> h, int child)
{
    while (child > 0)
    {
        int parent = (child - 1) / 2;
        if (h[parent] <= h[child]) return;
        (h[parent], h[child]) = (h[child], h[parent]);
        child = parent;
    }
}

static void SinkDown(List<int> h, int parent)
{
    while (true)
    {
        int left = 2 * parent + 1, right = left + 1, smallest = parent;
        if (left < h.Count && h[left] < h[smallest]) smallest = left;
        if (right < h.Count && h[right] < h[smallest]) smallest = right;
        if (smallest == parent) return;
        (h[parent], h[smallest]) = (h[smallest], h[parent]);
        parent = smallest;
    }
}

static bool IsValid(List<int> h)
{
    for (int c = 1; c < h.Count; c++) if (h[(c - 1) / 2] > h[c]) return false;
    return true;
}

start          [1, 3, 2, 7, 5, 6, 4]
index 4 -> 0   [0, 1, 2, 7, 3, 6, 4]  valid=True
index 0 -> 9   [1, 3, 2, 7, 9, 6, 4]  valid=True


Lowering index 4 from 5 to 0 sends the value up past 3 and then past 1 to the root, two swaps, and the heap stays valid. Raising the root to 9 sends it down: the smaller child 1 moves up, then 9 continues down past 3, and settles as a leaf. Both calls repaired one path, so each costs O(log n) once the index is known. The dictionary that supplies the index is what the library's `PriorityQueue<TElement, TPriority>` lacks; it offers no priority update, so applications that need one either keep their own indexed heap or enqueue a fresh entry and discard the stale one when it surfaces, which is the technique Chapter 24 uses for shortest paths.

```{index} Code Completion: Choose the Smaller Child
```

### Code Completion: Choose the Smaller Child

`Dequeue` below removes the root and moves the last item to the top, but the sink-down loop is incomplete: it never decides which child to compare with. Complete the two missing lines so that `smaller` names the child with the lower value when a right child exists. The expected output is `0 1 2 3 4 5 6 7`, each value on its own line.

In [ ]:
using System;
using System.Collections.Generic;

var heap = new List<int> { 0, 1, 2, 3, 5, 6, 4, 7 };
while (heap.Count > 0) Console.WriteLine(Dequeue(heap));

static int Dequeue(List<int> h)
{
    int minimum = h[0];
    h[0] = h[^1];
    h.RemoveAt(h.Count - 1);
    int parent = 0;
    while (2 * parent + 1 < h.Count)
    {
        int left = 2 * parent + 1, right = left + 1;
        int smaller = left;
        // Your code starts here.
        // If a right child exists and is smaller than the left child, choose it.

        // Your code ends here.
        if (h[parent] <= h[smaller]) break;
        (h[parent], h[smaller]) = (h[smaller], h[parent]);
        parent = smaller;
    }
    return minimum;
}

In [ ]:
// Solution
using System;
using System.Collections.Generic;

var heap = new List<int> { 0, 1, 2, 3, 5, 6, 4, 7 };
while (heap.Count > 0) Console.WriteLine(Dequeue(heap));

static int Dequeue(List<int> h)
{
    int minimum = h[0];
    h[0] = h[^1];
    h.RemoveAt(h.Count - 1);
    int parent = 0;
    while (2 * parent + 1 < h.Count)
    {
        int left = 2 * parent + 1, right = left + 1;
        int smaller = left;
        if (right < h.Count && h[right] < h[left]) smaller = right;
        if (h[parent] <= h[smaller]) break;
        (h[parent], h[smaller]) = (h[smaller], h[parent]);
        parent = smaller;
    }
    return minimum;
}

0
1
2
3
4
5
6
7


The starter already runs, and its output is wrong rather than crashing: it always compares with the left child, so it prints `0 1 3 4 5 6 2 7`. At the second removal the left child 3 is chosen although the right child 2 is smaller, 3 ends up above 2, and 2 is not reported until almost the end. That is the dangerous kind of defect, visible only on some inputs. The one-line fix tests the right child's existence before reading it; omitting `right < h.Count` throws `ArgumentOutOfRangeException` when the last parent has a single child.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. Bubble-up stops as soon as the parent is no larger. Why is it safe to stop there instead of checking the rest of the path?

```{dropdown} Answer
Every parent-child pair above that point was valid before the enqueue and has not been touched, and the pair just checked is valid. The only pair the enqueue could break was the new item against its parent, and that relation has now been repaired or was never broken.
```

2. Building a heap from 1,000,000 values takes under a million swaps bottom up but about 20 million with repeated enqueue. Where does the difference come from?

```{dropdown} Answer
From where the work happens. Enqueue moves items upward, and half of all items start as leaves with the longest possible path to the root. Bottom-up construction moves items downward, and half of the nodes are leaves that need no work at all; the few nodes high in the tree with long paths are a tiny fraction of n.
```

3. After `Dequeue`, the last item was moved to the root and sunk down. Why can it not need to bubble up as well?

```{dropdown} Answer
It came from a leaf position, where it was no smaller than every ancestor along its path, including the old root's children. Placed at the root it can only be too large for its new children, never too small for a parent it does not have.
```

4. An application needs to cancel a job that is somewhere in a heap of 50,000 jobs. What extra information makes that O(log n) rather than O(n)?

```{dropdown} Answer
The job's current index, kept in a dictionary that the heap updates on every swap. With the index known, cancellation is an overwrite with the last item plus one repair path; without it, the only way to find the job is a linear scan.
```

```{index} Practice: Check Every Parent
```

## Practice: Check Every Parent




```{index} Practice: Check Every Parent
```


Implement the invariant check for all non-root positions, using the heap's configured comparer.


In [ ]:
using System;
using System.Collections.Generic;

var heap=new BinaryMinHeap<int>();foreach(int value in new[]{4,1,3,1})heap.Enqueue(value);
while(heap.TryDequeue(out int next))Console.WriteLine(next);

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new List<T>();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            { /* TODO: reject a child that precedes its parent. */ }
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}


In [1]:
using System;
using System.Collections.Generic;

var heap=new BinaryMinHeap<int>();foreach(int value in new[]{4,1,3,1})heap.Enqueue(value);
while(heap.TryDequeue(out int next))Console.WriteLine(next);

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new List<T>();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}


1
1
3
4


The output is `1`, `1`, `3`, `4`. Both copies of 1 come out, in some order, before 3 and 4; duplicates are ordinary values to a heap. The invariant check runs against the same comparer the heap uses, so a heap built with a reversed comparer would validate under its own rule, not under `<=`.


```{rubric} Footnotes
```


[^1802_heap_implementation-1]: The teaching implementation is a binary min-heap. A library priority queue may use a different internal layout while honoring its public priority contract.


[^development-reading]: Further reading: [OpenDSA, Heaps and Priority Queues](https://opendsax.cs.vt.edu/ODSA/Books/Everything/html/Heaps.html). The worked business scenarios and guided reasoning here are original teaching examples.
